# RAG Avanzado No-Semántico — BM25+Chroma(pymupdf) + CrossEncoder + HyDE

Igual que RAG avanzado S2b pero usando **db_rgpd_pymupdf** en lugar de db_rgpd_semantic.

Pipeline:
1. **HyDE**: genera documento hipotético con Llama-3-8B (Ollama)
2. **EnsembleRetriever**: BM25 (0.4) + Chroma pymupdf (0.6), RRF, k=10
3. **CrossEncoder**: BAAI/bge-reranker-v2-m3 reranking top-10 → top-5
4. **Generador**: Llama-3-8B vía Ollama

**Orden de ejecución:**
1. Celda 1 — instalar deps → **Restart session**
2. Celda 2 — instalar y arrancar Ollama (tarda ~3-5 min)
3. Celda 3 en adelante

**Requisitos Drive:** `TFM_RGPD/db_rgpd_pymupdf/`  
**Subir al panel de archivos:** `dataset_test.json`

In [ ]:

# ── CELDA 2: Instalar zstd + Ollama + pull llama3:8b ─────────────────────────
import subprocess, time, os

print("Instalando zstd (dependencia de Ollama)...")
subprocess.run("apt-get install -y zstd", shell=True, capture_output=True)
print("zstd OK")

print("Instalando Ollama...")
result = subprocess.run(
    "curl -fsSL https://ollama.ai/install.sh | sh",
    shell=True, capture_output=True, text=True
)
print(result.stdout[-500:] if result.stdout else "(sin stdout)")
if result.returncode != 0:
    print("STDERR:", result.stderr[-300:])
    raise RuntimeError("Fallo instalando Ollama")

which = subprocess.run("which ollama", shell=True, capture_output=True, text=True)
OLLAMA = which.stdout.strip() or "/usr/local/bin/ollama"
print(f"Ollama en: {OLLAMA}")
assert os.path.exists(OLLAMA), f"No encontrado: {OLLAMA}"

print("Arrancando servidor Ollama en background...")
subprocess.Popen([OLLAMA, "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

print("Descargando llama3:8b (~4.7 GB)...")
subprocess.run([OLLAMA, "pull", "llama3:8b"])

test = subprocess.run([OLLAMA, "list"], capture_output=True, text=True)
print("Modelos:", test.stdout)
print("Ollama listo")


In [ ]:
# ── CELDA 2: Instalar y arrancar Ollama + pull llama3:8b ──────────────────────
# Ejecutar UNA SOLA VEZ. Tarda ~3-5 min en descargar el modelo.
import subprocess, time, os

print("Instalando Ollama...")
os.system("curl -fsSL https://ollama.ai/install.sh | sh")

print("Arrancando servidor Ollama en background...")
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

print("Descargando llama3:8b (~4.7 GB, puede tardar 3-5 min)...")
os.system("ollama pull llama3:8b")

# Verificar que responde
import subprocess
test = subprocess.run(["ollama", "list"], capture_output=True, text=True)
print("Modelos disponibles:", test.stdout)
print("Ollama listo")

In [ ]:
# ── CELDA 3: Montar Google Drive ──────────────────────────────────────────────
import os
from google.colab import drive
drive.mount('/content/drive')

DB_DIR       = "/content/drive/MyDrive/TFM_RGPD/db_rgpd_pymupdf"
OUTPUT_LOCAL = "/content/rag_advanced_no_semantic_inference_dataset.json"
OUTPUT_DRIVE = "/content/drive/MyDrive/TFM_RGPD/rag_advanced_no_semantic_inference_dataset.json"

if not os.path.exists(DB_DIR):
    raise FileNotFoundError(f"No se encuentra {DB_DIR}")
print("Drive OK")

In [ ]:
# ── CELDA 4: Cargar dataset ───────────────────────────────────────────────────
import json, re

DATASET_PATH = "/content/dataset_test.json"
if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError("Sube dataset_test.json al panel de archivos")

test_data = []
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip(): continue
        obj = json.loads(line)
        full_text = obj.get("text", "")
        pregunta_match = re.search(r'Pregunta:\s*(.*?)<\|eot_id\|>', full_text)
        gt_match = re.search(r'assistant<\|end_header_id\|>\n(.*?)(?:<\|eot_id\|>|$)', full_text, re.DOTALL)
        if pregunta_match and gt_match:
            test_data.append({
                "question":     pregunta_match.group(1).strip(),
                "ground_truth": gt_match.group(1).strip(),
            })

print(f"Dataset: {len(test_data)} preguntas")

In [ ]:
# ── CELDA 5: Construir retriever avanzado ─────────────────────────────────────
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from sentence_transformers import CrossEncoder

print("Cargando embeddings bge-m3 (CPU)...")
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3", model_kwargs={"device": "cpu"})

print("Cargando vectorstore pymupdf...")
vector_db = Chroma(persist_directory=DB_DIR, embedding_function=embeddings)

print("Preparando BM25...")
raw = vector_db.get()
bm25_docs = [Document(page_content=t, metadata=m) for t, m in zip(raw["documents"], raw["metadatas"])]
bm25 = BM25Retriever.from_documents(bm25_docs, k=10)

print("Cargando CrossEncoder...")
cross_encoder = CrossEncoder("BAAI/bge-reranker-v2-m3", device="cpu")

K_RRF, TOP_N = 60, 5

def retrieve_advanced(query):
    bm25_r = bm25.invoke(query)
    chroma_r = vector_db.similarity_search(query, k=10)
    scores, doc_map = {}, {}
    for rank, doc in enumerate(bm25_r):
        k = doc.page_content[:200]
        scores[k] = scores.get(k, 0.0) + 0.4 / (K_RRF + rank + 1)
        doc_map[k] = doc
    for rank, doc in enumerate(chroma_r):
        k = doc.page_content[:200]
        scores[k] = scores.get(k, 0.0) + 0.6 / (K_RRF + rank + 1)
        doc_map[k] = doc
    fused = [doc_map[k] for k in sorted(scores, key=scores.__getitem__, reverse=True)[:10]]
    pairs = [[query, doc.page_content] for doc in fused]
    ce_scores = cross_encoder.predict(pairs)
    ranked = sorted(zip(ce_scores, fused), key=lambda x: x[0], reverse=True)
    return [doc for _, doc in ranked[:TOP_N]]

print("Retriever no-semantico listo")

In [ ]:
# ── CELDA 6: Cargar LLM Ollama + definir HyDE ────────────────────────────────
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate

llm = OllamaLLM(model="llama3:8b", temperature=0.0)

# Test rápido
print("Test Ollama:", llm.invoke("Di 'OK' y nada más."))

TEMPLATE = """Eres un consultor juridico especializado en el RGPD. Responde de forma DIRECTA y CONCISA.
REGLAS:
- Usa UNICAMENTE la informacion del contexto proporcionado.
- Si el articulo exacto aparece en el contexto, citalo.
- Si la respuesta no esta en el contexto, responde: \"El contexto no contiene informacion suficiente.\"
- Responde siempre en espanol.

Contexto legal del RGPD:
{context}

Pregunta: {question}

Respuesta directa (maximo 3-4 frases):"""

def hyde_expand(question):
    prompt = (
        "Genera un parrafo breve de documentacion legal del RGPD que responderia "
        "directamente a esta pregunta. Solo el fragmento, sin preambulo.\n\n"
        f"Pregunta: {question}\nFragmento hipotetico:"
    )
    return llm.invoke(prompt).strip()

print("LLM listo")

In [ ]:
# ── CELDA 7: INFERENCIA ───────────────────────────────────────────────────────
results = []

for i, entrada in enumerate(test_data):
    pregunta = entrada["question"]
    print(f"[{i+1}/{len(test_data)}] Procesando...", flush=True)

    query_expandida = hyde_expand(pregunta)
    docs = retrieve_advanced(query_expandida)
    contextos = [doc.page_content for doc in docs]
    contexto_str = "\n\n".join(contextos)
    respuesta = llm.invoke(TEMPLATE.format(context=contexto_str, question=pregunta))

    results.append({
        "question":     pregunta,
        "answer":       respuesta,
        "contexts":     contextos,
        "ground_truth": entrada["ground_truth"],
    })

print("\nInferencia completada")

In [ ]:
# ── CELDA 8: Guardar resultados ───────────────────────────────────────────────
for path in [OUTPUT_LOCAL, OUTPUT_DRIVE]:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=4)

print(f"Guardado local:  {OUTPUT_LOCAL}")
print(f"Guardado Drive:  {OUTPUT_DRIVE}")
print(f"Total entradas:  {len(results)}")